# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# General
from typing import Tuple, Sequence, Dict, Union, Optional
import numpy as np
import json
import torch
import torch.nn as nn
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.tensorboard import SummaryWriter
import gerry
import pickle

# diffusion policy import
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
from diffusers.training_utils import EMAModel
from diffusers.optimization import get_scheduler

# Painting imports
import cv2
from style.diffusion_policy_gml.dataset import GmlDatasetNoSliding
from style.diffusion_policy_gml.network import compute_noise, compute_orig
import style.diffusion_policy_gml.network as network
from style.diffusion_policy_gml.network_transformer import Transformer1d, Transpose
import style.diffusion_policy_gml.utils as utils
import load_gml

In [ ]:
RUN_FOLDER = Path('runs/Jul08_01-18-24_eagle')

with open(RUN_FOLDER / 'network_kwargs.json', 'r') as f:
    network_kwargs_prelim = json.load(f)
PRED_HORIZON = network_kwargs_prelim['horizon']
PREDICTION_DIM = network_kwargs_prelim['output_dim']
NUM_DIFFUSION_ITERS = 100
device = torch.device('cuda')

print(f'{PRED_HORIZON=}, {PREDICTION_DIM=}, {NUM_DIFFUSION_ITERS=}')

## Load Dataset
(for normalization/un-normalization)

In [ ]:
# dataset_path = "data/gml_by_drawing_PRESERVE_ASPECT_CENTERED_003000.zarr"
dataset_path = "data/gml_by_stroke_PRESERVE_ASPECT_CENTERED_003000.zarr"

with gerry.Stopwatch("Loading dataset"):
    dataset = GmlDatasetNoSliding(
        dataset_path=dataset_path,
        sequence_length=PRED_HORIZON,
        action_delta=True,
        action_penlift=True,
        ignore_jump_actions=True,
        normalize=dict(obs=False, action=True),
        prescale_obs = (-3, 3),
        prescale_penlift=(0, 3),
        max_drawings=1,
        min_traj_length=15
    )
utils.drawing_lims = dict(x=(-3, 3), y=(-3, 3))

## Diffusion Setup

In [ ]:
# Noise scheduler
noise_scheduler = DDPMScheduler(
    num_train_timesteps=NUM_DIFFUSION_ITERS,
    beta_schedule='squaredcos_cap_v2',
    clip_sample=True,
    clip_sample_range=3,
    prediction_type='epsilon'
)

In [ ]:
# Load the network_kwargs
n_emb = 768
InputEmbedding = nn.Sequential(
    Transpose(1, 2),
    nn.Conv1d(3, 15, 5, padding=2),
    nn.ReLU(),
    nn.Conv1d(15, n_emb, 5, padding=2),
    nn.ReLU(),
    Transpose(1, 2)
)

_, network_kwargs = Transformer1d.FromJson(RUN_FOLDER / 'network_kwargs.json', InputEmbedding)
for k, v in network_kwargs.items():
    print(f'{k:<20}', v)

In [ ]:
def load_model(checkpoint=None):
    fname = f'ema_noise_pred_net_{checkpoint}.pth' if checkpoint is not None else 'ema_noise_pred_net.pth'
    ema_noise_pred_net = Transformer1d(**network_kwargs)
    ema_noise_pred_net.to(device)
    ema_noise_pred_net.load_state_dict(torch.load(RUN_FOLDER / fname))
    return ema_noise_pred_net

ema_noise_pred_net = load_model()

# Now do edits

## First load a drawing into the correct format

In [ ]:
# Read in a gml file
# basename = 'S_2'
# basename = 'batch_0'
basename = 'shapes3'
# basename = 'RSS_line_1'
# basename = 'georgia_tech_4'

root = Path('data') / 'custom_gmls'
outroot = Path('results') / 'gerry13y_edit'
outroot.mkdir(exist_ok=True, parents=True)
infile = root / f'{basename}.json'

drawing = load_gml.Drawing(infile, scale_behavior='PRESERVE_ASPECT_CENTERED')

In [ ]:
# Plot all round-trip stuff to make sure it looks correct
fig, axes = plt.subplots(1, 3, figsize=(10, 2))

for i in range(len(drawing.strokes)):
    axes[0].plot(drawing.strokes[i][:, 1], drawing.strokes[i][:, 2], 'k.', markersize=1)
axes[0].axis('equal');
axes[0].set_title('GML drawing')

## Now do diffusion on the loaded drawing

In [ ]:
t_start = 10
repeat = 100
seed = 8675309 + 9
guidance_weight = 4e4
# guidance_weight = 1e4

B = len(drawing.strokes)
x = torch.zeros(B, PRED_HORIZON, 3)
for i, stroke in enumerate(drawing.strokes):
    n = stroke.shape[0]
    while n > 128:
        stroke = stroke[::2]
        n = stroke.shape[0]
    xy = (stroke[:, 1:3] - 0.5) * 6
    x[i, :n, :2] = torch.tensor(dataset.normalize_obs(xy))
    x[i, n - 1, 2] = 1

x = x.to(device)
x_new = x * 1

def loss(y):
    l1 = torch.nn.MSELoss()(y, x)
    l2 = torch.nn.MSELoss()(torch.diff(y, axis=1), torch.diff(x, axis=1))
    # return l1 + 3 * l2
    return l1
guidance = network.guidance_fn(loss, weight=guidance_weight)

torch.manual_seed(seed)
history = [x_new * 1]
for _ in range(repeat):
    x_noisy = network.add_noise(x_new, t_start, noise_scheduler)
    x_noisy[:, :, 2] = x[:, :, 2]
    if len(history) == 1:
        x_noisy_init = x_noisy * 1
    x_new = network.eval_partial(ema_noise_pred_net, noise_scheduler, x_noisy, t_start, guidance=guidance)
    history.append(x_new * 1)
history = torch.stack(history, dim=0)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(8, 8), sharex=True, sharey=True)
# fig, axes = plt.subplots(3, 1, figsize=(8, 8), sharex=True, sharey=True)

def plot_x(ax, x, title):
    y = utils.kill_after_penlift(x.detach().cpu().numpy(), -1)
    for b in range(y.shape[0]):
        # dx = (b / y.shape[0] - 0.4) * 4
        dx = 0
        utils.plot_traj(ax, y[b, :, 2], obs=y[b, :, :2] + [dx, 0], markersize=1)
    ax.set_title(title)

plot_x(axes[0], x, 'Original Trajectory')
plot_x(axes[1], x_noisy_init, 'Noisy Trajectory')
plot_x(axes[2], x_noisy, 'Noisy Trajectory')
plot_x(axes[3], x_new, 'Denoised Trajectory')
for ax in axes: ax.grid(False)
fig.tight_layout()

if True:
    fig.savefig(
        outroot /
        f'{basename}_out_t{t_start:02d}_r{repeat}_g{guidance_weight}_s{seed}.svg'
    )
    np.savez(
        outroot /
        f'{basename}_out_t{t_start:02d}_r{repeat}_g{guidance_weight}_s{seed}.npz',
        x=x.detach().cpu().numpy(),
        x_noisy=x_noisy.detach().cpu().numpy(),
        x_noisy_init=x_noisy_init.detach().cpu().numpy(),
        x_out=x_new.detach().cpu().numpy())


In [ ]:
r, c = 8, 3
# fig, axes = plt.subplots(8, 3, figsize=(16, 16), sharex=True, sharey=True)
fig, axes = plt.subplots(4, 6, figsize=(16, 16), sharex=True, sharey=True)

inds_to_plot = np.linspace(0, len(history) - 1, r * c - 1).astype(int)
for ax, i, xplot in zip(axes.flatten(), inds_to_plot, history[inds_to_plot]):
    plot_x(ax, xplot, f'After {i} iterations')
plot_x(axes[-1][-1], history[-1], 'Denoised Trajectory')
for ax in axes.flatten(): ax.grid(False)
fig.tight_layout()

if True:
    fig.savefig(
        outroot /
        f'{basename}_out_t{t_start:02d}_r{repeat}_g{guidance_weight}_s{seed}_progression.svg')
    np.savez(
        outroot /
        f'{basename}_out_t{t_start:02d}_r{repeat}_g{guidance_weight}_s{seed}_progression.npz',
        history=history.detach().cpu().numpy())